# EII — Phase 3: Segment Decomposition Analysis

Computes the EII for each of the six individual border segments of every
hexagonal cell, producing a directional decomposition of interface connectivity.

**Geometric properties (confirmed by diagnostic):**
- Each hexagon has exactly 6 segments of equal length (8,773.8 m)
- Segment orientations: 0°, 60°, 120°, 180°, 240°, 300° (perfect isotropy)
- Opposite pairs: (seg1,seg4), (seg2,seg5), (seg3,seg6)
- EII aggregated = arithmetic mean of 6 components (equal lengths)

**Outputs per cell per year:**
- `w1`–`w6`: EII per segment (ordered by vertex index, 0-based)
- `eii_mean`: aggregated EII = mean(w1..w6)
- `eii_sum`: total connected interface in segment units [0, 6]
- `aniso`: connectivity anisotropy = SD(w1..w6)
- `grad_EW`: E-W gradient = w5 (0°) − w2 (180°)
- `grad_NESW`: NE-SW gradient = w4 (60°) − w1 (240°)
- `grad_NWSE`: NW-SE gradient = w3 (120°) − w6 (300°)
- `grad_max`: magnitude of dominant directional gradient
- `grad_dir`: direction of dominant gradient (degrees)

---
## 1. Configuration — edit only this cell

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Primary hexagonal grid shapefile (HEX-20)
GRID_SHAPEFILE = r"D:\Modelo_LAPIG\grids\hex_20000ha.shp"

# Folder containing annual binary rasters (reclass_YYYY.tif)
RASTER_FOLDER = r"D:\Modelo_LAPIG\rasters_binarios"

# Output folder
OUTPUT_FOLDER = r"D:\Modelo_LAPIG\phase3_segments"

# Checkpoint folder
CHECKPOINT_FOLDER = r"D:\Modelo_LAPIG\phase3_segments\checkpoints"

# Nodata value (hardcoded — raster metadata declares nodata=0
# but 0 = non-natural vegetation, a valid value)
NODATA = 255

# Natural vegetation pixel value
VEGETATION_VALUE = 1

# Scenario map
SCENARIO_MAP = {
    "reclass": "OBS",
    "obs": "OBS",
}

---
## 2. Dependencies

In [ ]:
import importlib, subprocess, sys
for pkg in ["rasterstats", "geopandas", "rasterio", "pandas", "numpy",
            "matplotlib", "shapely"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"OK {pkg}")

---
## 3. Utility functions

In [ ]:
import os, re, glob, warnings, time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import LineString

YEAR_RE = re.compile(r"(19|20)\d{2}")

def extract_year(filename):
    m = YEAR_RE.search(filename)
    if not m:
        raise ValueError(f"Year not found in: {filename}")
    return m.group(0)

def infer_scenario(filename, scenario_map):
    lower = filename.lower()
    for key, label in scenario_map.items():
        if key in lower:
            return label
    return "OBS"

def ensure_id(gdf):
    if "ID_UNICO" not in gdf.columns:
        gdf = gdf.copy()
        gdf["ID_UNICO"] = gdf.index.astype(int)
    return gdf

def reproject_if_needed(gdf, raster_path):
    with rasterio.open(raster_path) as src:
        crs_raster = src.crs
    if crs_raster and gdf.crs != crs_raster:
        return gdf.to_crs(crs_raster)
    return gdf

print("Utility functions loaded.")

---
## 4. Segment extraction

Each hexagonal cell is decomposed into 6 individual LineString segments.
Segments are ordered by vertex index (0-based) and labeled with their
orientation angle.

Confirmed geometry (from diagnostic):
- Segment 1 (index 0→1): 240°
- Segment 2 (index 1→2): 180°
- Segment 3 (index 2→3): 120°
- Segment 4 (index 3→4):  60°
- Segment 5 (index 4→5):   0°
- Segment 6 (index 5→6): 300°

Opposite pairs for directional gradients:
- E-W axis:   seg5 (0°) ↔ seg2 (180°)
- NE-SW axis: seg4 (60°) ↔ seg1 (240°)
- NW-SE axis: seg3 (120°) ↔ seg6 (300°)

In [ ]:
# Segment angle labels (confirmed by diagnostic)
SEGMENT_ANGLES = [240, 180, 120, 60, 0, 300]  # degrees, seg1..seg6

# Opposite pairs for gradient computation: (positive_seg_idx, negative_seg_idx)
# gradient = w[pos] - w[neg]; positive direction is the first angle
GRADIENT_AXES = {
    "grad_EW":   (4, 1),   # seg5 (0°) − seg2 (180°)  → positive = eastward
    "grad_NESW": (3, 0),   # seg4 (60°) − seg1 (240°) → positive = NE direction
    "grad_NWSE": (2, 5),   # seg3 (120°) − seg6 (300°) → positive = NW direction
}


def extract_segments(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Decompose each hexagonal polygon into 6 individual LineString segments.

    Returns a GeoDataFrame with columns:
      ID_UNICO: cell identifier
      seg_idx:  segment index (0–5)
      angle:    segment orientation in degrees
      geometry: LineString for that segment
    """
    records = []
    for _, row in gdf.iterrows():
        coords = list(row.geometry.exterior.coords)
        # coords has 7 points: 6 unique + repeat of first
        for seg_idx in range(6):
            seg = LineString([coords[seg_idx], coords[seg_idx + 1]])
            records.append({
                "ID_UNICO": row["ID_UNICO"],
                "seg_idx":  seg_idx,
                "angle":    SEGMENT_ANGLES[seg_idx],
                "geometry": seg,
            })

    gdf_segs = gpd.GeoDataFrame(records, crs=gdf.crs)
    return gdf_segs


print("Segment extraction function loaded.")

---
## 5. EII extraction per segment

For each raster year, run zonal_stats on each of the 6×N segment
geometries and pivot to produce one row per cell with 6 EII columns.

In [ ]:
def compute_derived(df_wide: pd.DataFrame) -> pd.DataFrame:
    """
    Given a wide DataFrame with columns w1..w6 (EII per segment),
    add derived metrics:
      eii_mean, eii_sum, aniso, grad_EW, grad_NESW, grad_NWSE,
      grad_max, grad_dir
    """
    seg_cols = [f"w{i+1}" for i in range(6)]
    W = df_wide[seg_cols].values  # shape: (n_cells, 6)

    df_wide["eii_mean"] = np.round(np.nanmean(W, axis=1), 4)
    df_wide["eii_sum"]  = np.round(np.nansum(W, axis=1), 4)
    df_wide["aniso"]    = np.round(np.nanstd(W, axis=1), 4)

    # Directional gradients
    for name, (pos_idx, neg_idx) in GRADIENT_AXES.items():
        df_wide[name] = np.round(W[:, pos_idx] - W[:, neg_idx], 4)

    # Dominant gradient: axis with largest |gradient|
    grad_cols = list(GRADIENT_AXES.keys())
    grad_angles = [0, 60, 120]  # positive direction angles for EW, NESW, NWSE
    G = df_wide[grad_cols].values.astype(float)  # (n_cells, 3)

    max_idx = np.argmax(np.abs(G), axis=1)
    df_wide["grad_max"] = np.round(
        G[np.arange(len(G)), max_idx], 4)
    df_wide["grad_dir"] = [
        grad_angles[i] if G[r, i] >= 0
        else (grad_angles[i] + 180) % 360
        for r, i in enumerate(max_idx)
    ]
    return df_wide


def extract_segments_eii(gdf_segs: gpd.GeoDataFrame,
                          raster_path: str,
                          checkpoint_folder: str) -> pd.DataFrame:
    """
    Extract EII for all 6 segments of all cells for one raster.

    Returns wide DataFrame: ID_UNICO, w1..w6, eii_mean, eii_sum,
    aniso, grad_EW, grad_NESW, grad_NWSE, grad_max, grad_dir
    """
    base     = os.path.basename(raster_path)
    year     = extract_year(base)
    scenario = infer_scenario(base, SCENARIO_MAP)
    tag      = f"{scenario}_{year}"

    ckpt = os.path.join(checkpoint_folder, f"seg_{tag}.csv")
    if os.path.exists(ckpt):
        print(f"    [ckpt] {tag}")
        return pd.read_csv(ckpt)

    print(f"    [run]  {tag}")
    t0 = time.time()

    # Reproject segments to raster CRS
    gdf_proj = reproject_if_needed(gdf_segs, raster_path)

    # Run zonal_stats on all 6×N segment geometries at once
    stats = zonal_stats(
        gdf_proj, raster_path,
        categorical=True,
        nodata=NODATA,
        all_touched=True,
    )

    # Compute EII per segment row
    px_veg = np.array(
        [s.get(VEGETATION_VALUE, 0) or 0 for s in stats], dtype=float)
    px_tot = np.array(
        [sum(v for k, v in s.items() if k != NODATA) for s in stats],
        dtype=float)

    with np.errstate(invalid="ignore", divide="ignore"):
        eii_per_seg = np.where(px_tot > 0, px_veg / px_tot, np.nan)

    # Assemble long DataFrame
    df_long = pd.DataFrame({
        "ID_UNICO": gdf_segs["ID_UNICO"].values,
        "seg_idx":  gdf_segs["seg_idx"].values,
        "eii":      np.round(eii_per_seg, 4),
    })

    # Pivot to wide: one row per cell, columns w1..w6
    df_wide = df_long.pivot(
        index="ID_UNICO", columns="seg_idx", values="eii"
    ).reset_index()
    df_wide.columns = ["ID_UNICO"] + [f"w{i+1}" for i in range(6)]

    # Add derived metrics
    df_wide = compute_derived(df_wide)

    df_wide.to_csv(ckpt, index=False)
    print(f"           done in {time.time()-t0:.0f}s")
    return df_wide


print("EII segment extraction function loaded.")

---
## 6. Load grid and prepare segments

In [ ]:
os.makedirs(OUTPUT_FOLDER,     exist_ok=True)
os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)

print("Loading HEX-20 grid...")
gdf = gpd.read_file(GRID_SHAPEFILE)
gdf = ensure_id(gdf)
print(f"  {len(gdf):,} cells | CRS: {gdf.crs}")

print("\nExtracting 6 segments per cell...")
gdf_segs = extract_segments(gdf)
print(f"  {len(gdf_segs):,} segment geometries ({len(gdf_segs)//6:,} cells × 6)")

# Quick sanity check on segment geometry
lengths = gdf_segs.geometry.length
print(f"  Segment length: min={lengths.min():.1f}m  "
      f"max={lengths.max():.1f}m  mean={lengths.mean():.1f}m")
if lengths.max() - lengths.min() < 1.0:
    print("  OK: all segments have equal length (hexagons are regular)")
else:
    print("  WARNING: segments have unequal lengths — check geometry")

# List rasters
rasters = sorted(glob.glob(os.path.join(RASTER_FOLDER, "*.tif")))
years   = [extract_year(os.path.basename(r)) for r in rasters]
print(f"\n{len(rasters)} rasters: {min(years)} to {max(years)}")

---
## 7. Run annual segment pipeline

Processes all annual rasters. Each year produces one checkpoint CSV
with columns: ID_UNICO, w1..w6, eii_mean, eii_sum, aniso,
grad_EW, grad_NESW, grad_NWSE, grad_max, grad_dir.

Safe to interrupt and resume — completed years load from checkpoint.

In [ ]:
results_by_year = {}  # {year_str: DataFrame}

n_total    = len(rasters)
n_computed = 0
n_ckpt     = 0
t_start    = time.time()

print(f"Starting segment pipeline — {n_total} rasters")
print("-" * 55)

for i, raster_path in enumerate(rasters, 1):
    year = extract_year(os.path.basename(raster_path))
    ckpt = os.path.join(CHECKPOINT_FOLDER, f"seg_OBS_{year}.csv")

    df = extract_segments_eii(gdf_segs, raster_path, CHECKPOINT_FOLDER)
    results_by_year[year] = df

    if os.path.exists(ckpt):
        n_ckpt += 1
    else:
        n_computed += 1
        elapsed = time.time() - t_start
        remaining = (elapsed / i) * (n_total - i)
        print(f"  [{i:02d}/{n_total}] {year} | "
              f"est. remaining: {remaining/60:.0f} min")

print("-" * 55)
print(f"Done. Computed: {n_computed} | From checkpoint: {n_ckpt}")
print(f"Total time: {(time.time()-t_start)/60:.1f} min")

---
## 8. Consolidate and export

In [ ]:
# Build consolidated wide datasets — one column per metric per year
metrics = ["eii_mean", "eii_sum", "aniso",
           "grad_EW", "grad_NESW", "grad_NWSE",
           "grad_max", "grad_dir"]
seg_cols = [f"w{i+1}" for i in range(6)]

df_base = gdf[["ID_UNICO"]].copy()

# Separate consolidated CSVs per metric family
dfs = {m: df_base.copy() for m in metrics}
df_segs_all = df_base.copy()  # all w1..w6 across years

for year in sorted(results_by_year.keys()):
    df = results_by_year[year]
    for m in metrics:
        dfs[m] = dfs[m].merge(
            df[["ID_UNICO", m]].rename(columns={m: f"{m}_{year}"}),
            on="ID_UNICO", how="left")
    for sc in seg_cols:
        df_segs_all = df_segs_all.merge(
            df[["ID_UNICO", sc]].rename(columns={sc: f"{sc}_{year}"}),
            on="ID_UNICO", how="left")

# Save
for m, df_m in dfs.items():
    path = os.path.join(OUTPUT_FOLDER, f"seg_{m}_annual.csv")
    df_m.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {os.path.basename(path)}  shape={df_m.shape}")

seg_path = os.path.join(OUTPUT_FOLDER, "seg_w1w6_annual.csv")
df_segs_all.to_csv(seg_path, index=False, encoding="utf-8-sig")
print(f"Saved: seg_w1w6_annual.csv  shape={df_segs_all.shape}")

---
## 9. Analysis 1 — Connectivity anisotropy

Spatial map and temporal trend of SD(w1..w6) per cell.

In [ ]:
import matplotlib.pyplot as plt

years_sorted = sorted(results_by_year.keys())
years_int    = [int(y) for y in years_sorted]

# Annual domain mean anisotropy
aniso_mean = [results_by_year[y]["aniso"].mean() for y in years_sorted]
aniso_p75  = [results_by_year[y]["aniso"].quantile(0.75) for y in years_sorted]
aniso_p25  = [results_by_year[y]["aniso"].quantile(0.25) for y in years_sorted]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: temporal trend
ax = axes[0]
ax.fill_between(years_int, aniso_p25, aniso_p75,
                alpha=0.25, color="steelblue", label="IQR")
ax.plot(years_int, aniso_mean, color="steelblue",
        linewidth=2, label="Mean anisotropy")
ax.set_xlabel("Year")
ax.set_ylabel("Anisotropy SD(w1..w6)")
ax.set_title("Temporal trend of connectivity anisotropy")
ax.legend()
ax.grid(True, alpha=0.3)

# Panel B: spatial map for 2024 (or most recent year)
last_year = years_sorted[-1]
gdf_map = gdf[["ID_UNICO", "geometry"]].merge(
    results_by_year[last_year][["ID_UNICO", "aniso"]],
    on="ID_UNICO")
gdf_map.plot(column="aniso", ax=axes[1], cmap="YlOrRd",
             legend=True,
             legend_kwds={"label": "Anisotropy SD"},
             linewidth=0.1, edgecolor="none")
axes[1].set_title(f"Spatial distribution of anisotropy — {last_year}")
axes[1].axis("off")

plt.suptitle("Connectivity anisotropy — HEX-20 segment decomposition",
             fontsize=12)
plt.tight_layout()
fig_path = os.path.join(OUTPUT_FOLDER, "aniso_trend_and_map.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

# Summary table
print("\nAnisotropy summary (first and last 5 years):")
summary = pd.DataFrame({
    "year": years_int,
    "aniso_mean": [round(v, 4) for v in aniso_mean],
    "aniso_p25":  [round(v, 4) for v in aniso_p25],
    "aniso_p75":  [round(v, 4) for v in aniso_p75],
})
print(pd.concat([summary.head(5), summary.tail(5)]).to_string(index=False))
summary.to_csv(os.path.join(OUTPUT_FOLDER, "aniso_annual_summary.csv"),
               index=False, encoding="utf-8-sig")

---
## 10. Analysis 2 — Directional gradients

Maps of the three directional gradients and the dominant gradient
direction for selected years.

In [ ]:
# Select 3 representative years
selected = [years_sorted[0], years_sorted[len(years_sorted)//2], years_sorted[-1]]
grad_names = ["grad_EW", "grad_NESW", "grad_NWSE"]
grad_labels = ["E-W gradient (0°-180°)",
               "NE-SW gradient (60°-240°)",
               "NW-SE gradient (120°-300°)"]

for year in selected:
    df_y = results_by_year[year]
    gdf_y = gdf[["ID_UNICO", "geometry"]].merge(
        df_y[["ID_UNICO"] + grad_names + ["grad_max"]],
        on="ID_UNICO")

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    vmax = max(gdf_y[g].abs().quantile(0.95) for g in grad_names)

    for ax, grad, label in zip(axes[:3], grad_names, grad_labels):
        gdf_y.plot(column=grad, ax=ax, cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax,
                   legend=True,
                   legend_kwds={"label": "Gradient (EII)"},
                   linewidth=0.1, edgecolor="none")
        ax.set_title(label, fontsize=9)
        ax.axis("off")

    # Panel 4: dominant gradient magnitude
    gdf_y.plot(column="grad_max", ax=axes[3], cmap="RdBu_r",
               vmin=-vmax, vmax=vmax,
               legend=True,
               legend_kwds={"label": "Max gradient"},
               linewidth=0.1, edgecolor="none")
    axes[3].set_title("Dominant gradient direction", fontsize=9)
    axes[3].axis("off")

    plt.suptitle(f"Directional connectivity gradients — {year}", fontsize=12)
    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_FOLDER, f"gradients_{year}.png")
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}")

# Annual mean gradient magnitudes
grad_summary = pd.DataFrame({"year": years_int})
for g in grad_names:
    grad_summary[f"{g}_mean"] = [
        round(results_by_year[y][g].mean(), 4) for y in years_sorted]
    grad_summary[f"{g}_abs_mean"] = [
        round(results_by_year[y][g].abs().mean(), 4) for y in years_sorted]

grad_path = os.path.join(OUTPUT_FOLDER, "gradients_annual_summary.csv")
grad_summary.to_csv(grad_path, index=False, encoding="utf-8-sig")
print(f"\nGradient summary saved: {grad_path}")
print(grad_summary.head(5).to_string(index=False))

---
## 11. Quality control — verify consistency with aggregated EII

The mean of the 6 segment EII values must equal the aggregated EII
from Phase 2 (within floating-point tolerance).
Any systematic difference indicates a bug in either pipeline.

In [ ]:
# Load Phase 2 aggregated EII for comparison
EII_ANNUAL_PATH = r"D:\Modelo_LAPIG\phase2_annual\eii_HEX20_annual.csv"

try:
    df_eii_phase2 = pd.read_csv(EII_ANNUAL_PATH)
    print("Phase 2 EII loaded for cross-validation.")

    max_diffs = []
    for year in years_sorted:
        # Find the EII column for this year in Phase 2
        eii_col = [c for c in df_eii_phase2.columns
                   if year in c and "eii" in c.lower()]
        if not eii_col:
            continue

        df_p2 = df_eii_phase2[["ID_UNICO", eii_col[0]]].rename(
            columns={eii_col[0]: "eii_phase2"})
        df_p3 = results_by_year[year][["ID_UNICO", "eii_mean"]]

        merged = df_p2.merge(df_p3, on="ID_UNICO")
        diff = (merged["eii_mean"] - merged["eii_phase2"]).abs()
        max_diff = diff.max()
        max_diffs.append({"year": year, "max_abs_diff": round(max_diff, 6)})

    df_diff = pd.DataFrame(max_diffs)
    overall_max = df_diff["max_abs_diff"].max()
    print(f"\nMax absolute difference (Phase 2 vs Phase 3): {overall_max:.6f}")
    if overall_max < 0.001:
        print("OK: segment mean matches aggregated EII within tolerance.")
    else:
        print("WARNING: difference exceeds 0.001 — investigate pipeline.")
    print(df_diff.to_string(index=False))

except FileNotFoundError:
    print("Phase 2 EII file not found — skipping cross-validation.")
    print(f"Expected: {EII_ANNUAL_PATH}")